# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:
def to_num(s):
    return float(s.replace(" ", ""))
latin = {
    "almaty": "алматы",
    "astana": "астана",
    "shymkent": "шымкент",
    "karaganda": "караганда",
    "aktobe": "актобе",
    "atyrau": "атырау",
}

def clean_city(s):
    s = s.strip().lower()
    s = latin.get(s, s)
    return s.title()


print(to_num("9 610"), to_num("8070.0"))
print(clean_city(" Алматы"), clean_city("Astana "), clean_city("АЛМАТЫ"), clean_city("Shymkent"))
print(sorted({clean_city(r["city"]) for r in rows}))

9610.0 8070.0
Алматы Астана Алматы Шымкент
['Актобе', 'Алматы', 'Астана', 'Атырау', 'Караганда', 'Шымкент']


In [4]:
cats = {}
for r in rows:
    cats[r["category"]] = cats.get(r["category"], 0) + 1

for c, n in sorted(cats.items(), key=lambda x: x[1], reverse=True):
    print(f"{c:<12} {n:>5}")

Одежда        1529
Электроника   1185
Дом           1105
Красота        991
Книги          653


In [5]:
rev = {}
cnt = {}
for r in rows:
    if r["status"] == "оплачен":
        c = r["category"]
        rev[c] = rev.get(c, 0) + to_num(r["amount"])
        cnt[c] = cnt.get(c, 0) + 1

print(f"{'Категория':<12} | {'Выручка, ₸':>14} | {'Средний чек, ₸':>14}")
print("-" * 46)
for c, total in sorted(rev.items(), key=lambda x: x[1], reverse=True):
    print(f"{c:<12} | {total:>14,.0f} | {total / cnt[c]:>14,.0f}")

Категория    |     Выручка, ₸ | Средний чек, ₸
----------------------------------------------
Электроника  |     65,270,240 |         61,287
Одежда       |     18,321,080 |         13,286
Дом          |     10,516,320 |         10,655
Красота      |      5,507,620 |          6,188
Книги        |      2,294,100 |          3,935


In [6]:
months = {}
for r in rows:
    m = r["order_date"][:7]
    months[m] = months.get(m, 0) + 1

for m in sorted(months):
    print(m, months[m])

2026-09 2782
2026-10 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [7]:
month_rev = {}
for r in rows:
    if r["status"] == "оплачен":
        m = r["order_date"][:7]
        month_rev[m] = month_rev.get(m, 0) + to_num(r["amount"])

for m in sorted(month_rev):
    print(f"{m}: {month_rev[m]:,.0f} ₸")

sep = month_rev["2026-09"]
oct_ = month_rev["2026-10"]
print(f"Изменение октября к сентябрю: {(oct_ - sep) / sep * 100:+.1f}%")

2026-09: 57,489,850 ₸
2026-10: 44,419,510 ₸
Изменение октября к сентябрю: -22.7%


In [8]:
total = {}
canceled = {}
for r in rows:
    city = clean_city(r["city"])
    total[city] = total.get(city, 0) + 1
    if r["status"] == "отменён":
        canceled[city] = canceled.get(city, 0) + 1

for city in sorted(total, key=lambda c: total[c], reverse=True):
    k = canceled.get(city, 0)
    print(f"{city:<10} {k / total[city] * 100:.1f}%  ({k} из {total[city]})")

Алматы     7.9%  (172 из 2188)
Астана     8.8%  (123 из 1390)
Шымкент    20.4%  (164 из 804)
Караганда  8.9%  (48 из 538)
Актобе     10.3%  (41 из 397)
Атырау     7.5%  (11 из 146)


In [9]:
clients = {}
for r in rows:
    if r["status"] == "оплачен":
        cid = r["customer_id"]
        clients[cid] = clients.get(cid, 0) + to_num(r["amount"])

top = sorted(clients.items(), key=lambda x: x[1], reverse=True)[:5]
for i, (cid, s) in enumerate(top, start=1):
    print(f"{i}. {cid:<8} {s:>14,.0f} ₸")

1. TEST         29,999,970 ₸
2. C01627          257,180 ₸
3. C00485          215,920 ₸
4. C01338          192,080 ₸
5. C02122          191,670 ₸


In [10]:
for r in rows:
    if r["customer_id"] == top[0][0]:
        print(r["order_date"], clean_city(r["city"]), r["category"], r["amount"], r["status"])

2026-09-12 11:00 Алматы Электроника 9999990.0 оплачен
2026-09-25 15:00 Алматы Электроника 9999990.0 оплачен
2026-10-03 10:00 Алматы Электроника 9999990.0 оплачен


Ответ к 7 задаче: на первом месте клиент `TEST`: 29,999,970 тг и это не настоящий покупатель, а тестовый. У него три одинаковых заказа по 9999990.0 тг.  Перед расчётом метрик такие строки нужно убрать

In [ ]:
uniq = {}
for r in rows:
    city = clean_city(r["city"])
    if city not in uniq:
        uniq[city] = set()
    uniq[city].add(r["customer_id"])
for city in sorted(uniq, key=lambda c: len(uniq[c]), reverse=True):
    print(f"{city:<10} {len(uniq[city]):>5}")

Алматы      1351
Астана       981
Шымкент      655
Караганда    451
Актобе       358
Атырау       138


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [12]:
# Челлендж